In [12]:
import json
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer 
import numpy as np 

Experiments 1

In [9]:
documents = [] 
with open('D:\\VietAnd\\Việt Anh\\NLP_HUS\\c4-train.00000-of-01024-30K.json\\c4-train.00000-of-01024-30K.json', 'r', encoding = 'utf-8') as f: 
    for line in f: 
        if line.strip(): 
            data = json.loads(line) 
            documents.append(data['text']) 

print(f"Number of documents: {len(documents)}")
print(f"First document: {documents[0]}")

Number of documents: 30000
First document: Beginners BBQ Class Taking Place in Missoula!
Do you want to get better at making delicious BBQ? You will have the opportunity, put this on your calendar now. Thursday, September 22nd join World Class BBQ Champion, Tony Balay from Lonestar Smoke Rangers. He will be teaching a beginner level class for everyone who wants to get better with their culinary skills.
He will teach you everything you need to know to compete in a KCBS BBQ competition, including techniques, recipes, timelines, meat selection and trimming, plus smoker and fire information.
The cost to be in the class is $35 per person, and for spectators it is free. Included in the cost will be either a t-shirt or apron and you will be tasting samples of each meat that is prepared.


In [10]:
vectorizer = CountVectorizer() 
X_count = vectorizer.fit_transform(documents) 
tfidf_transformer = TfidfVectorizer() 
X_tfidf = tfidf_transformer.fit_transform(documents) 

feature_names = vectorizer.get_feature_names_out() 

N, V = X_tfidf.shape 
print(f"Number of documents (N): {N}")
print(f"Number of features (V): {V}")
print(f"Matrix shape: {X_tfidf.shape}") 

Number of documents (N): 30000
Number of features (V): 193540
Matrix shape: (30000, 193540)


In [11]:
# tính tỷ lệ sparsity 
nnz = X_tfidf.nnz 

sparsity = 1 - (nnz / (N * V)) 
print(f"số phần tử khác 0: {nnz:,}") 
print(f"tổng số phần tử (N * V): {N * V:,}") 
print(f"sparsity = {sparsity:.4f}") 


số phần tử khác 0: 4,985,822
tổng số phần tử (N * V): 5,806,200,000
sparsity = 0.9991


Câu hỏi: Tại sao một document chỉ sử dụng một phần rất nhỏ vocabulary nhưng vector vẫn có chiều 
V
Trả lời: Trong mô hình Không gian Vector (Vector Space Model), để có thể so sánh toán học (như tính Cosine Similarity) giữa bất kỳ hai document nào, tất cả các document phải được ánh xạ vào cùng một hệ trục tọa độ chung. Hệ trục này chính là Vocabulary kích thước V, trong đó mỗi từ là một chiều (dimension) độc lập. Dù một document chỉ chứa vài chục từ (rất nhỏ so với V), nó vẫn phải giữ nguyên độ dài vector là V với các giá trị 0 ở những từ không xuất hiện. Nếu không có chung chiều V, ta không thể thực hiện phép nhân ma trận (dot product) để tìm sự tương đồng.

In [13]:
# tìm 20 term phổ biến nhất theo document frequency 
# tìm 20 term có idf cao nhất 
# tìm 20 term có tf-idf cao nhất trong một document được chọn 

df_array = np.array((X_count > 0).sum(axis=0)).flatten() 
top_20 = df_array.argsort()[::-1][:20] 
print("top 20 term theo document frequency") 
for id in top_20: 
    print(f"{feature_names[id]}: {df_array[id]}") 

idf_array = tfidf_transformer.idf_ 
top_20_idf = idf_array.argsort()[::-1][:20]
print("top 20 term theo idf")   
for id in top_20_idf: 
    print(f"{feature_names[id]}: {idf_array[id]}")

doc_index = 0 
doc_tfidf = X_tfidf[doc_index].toarray().flatten() 
top_20_tfidf_indices = doc_tfidf.argsort()[::-1][:20]
print(f"top 20 term theo tf-idf trong document {doc_index}")
print(f"\n--- 3. Top 20 terms có TF-IDF cao nhất trong Document {doc_index} ---")
for idx in top_20_tfidf_indices:
        if doc_tfidf[idx] > 0:
          print(f"{feature_names[idx]:<15}: TF-IDF = {doc_tfidf[idx]:.4f}")


top 20 term theo document frequency
the: 27893
and: 27423
to: 26689
of: 26031
in: 25224
for: 23651
is: 22739
with: 21405
on: 20262
that: 18370
this: 17840
are: 17594
it: 17168
as: 16467
at: 16347
from: 16316
be: 16153
you: 16094
by: 15123
have: 14852
top 20 term theo idf
00000: 10.615838812862137
00003: 10.615838812862137
000040: 10.615838812862137
00005: 10.615838812862137
0000856166: 10.615838812862137
0001042: 10.615838812862137
000116: 10.615838812862137
00012: 10.615838812862137
00015: 10.615838812862137
00016: 10.615838812862137
000165101: 10.615838812862137
0002: 10.615838812862137
00022: 10.615838812862137
000226: 10.615838812862137
000281: 10.615838812862137
확인하게: 10.615838812862137
환원되지: 10.615838812862137
활동: 10.615838812862137
활동에: 10.615838812862137
활동을: 10.615838812862137
top 20 term theo tf-idf trong document 0

--- 3. Top 20 terms có TF-IDF cao nhất trong Document 0 ---
bbq            : TF-IDF = 0.4618
class          : TF-IDF = 0.2758
meat           : TF-IDF = 0.1942
ba

Câu hỏi 1: Một term xuất hiện rất nhiều trong corpus có nhất thiết có TF-IDF cao không?

Trả lời: Không. Các term xuất hiện trong hầu hết corpus (top DF) thường là stopwords (như the, a, and, to, is...).
Câu hỏi 2: Một term có IDF cao có nhất thiết có TF-IDF cao trong mọi document không?

Trả lời: Không. Một term có IDF cao chứng tỏ nó rất hiếm (hiếm gặp trong toàn bộ corpus)


PART E 

In [14]:
corpus = [
    "cat eats fish",  # D1
    "dog eats fish",  # D2
    "cat likes fish"  # D3
]
def build_vocabulary(docs): 
    vocab = set() 
    for doc in docs: 
        tokens = doc.split() 
        vocab.update(tokens) 
    return sorted(vocab) 

vocab = build_vocabulary(corpus) 
print(f"Vocabulary: {vocab}") 

def compute_counts(docs, vocab):
    counts = np.zeros((len(docs), len(vocab)))
    vocab_dict = {word: i for i, word in enumerate(vocab)}
    
    for i, doc in enumerate(docs):
        for word in doc.split():
            if word in vocab_dict:
                counts[i, vocab_dict[word]] += 1
                
    return counts
counts_matrix = compute_counts(corpus, vocab)
print("\nCount Matrix:\n", counts_matrix)

def compute_tf(counts):
    doc_lengths = counts.sum(axis=1, keepdims=True)
    tf_matrix = np.divide(counts, doc_lengths, out=np.zeros_like(counts), where=doc_lengths!=0)
    
    return tf_matrix

tf_matrix = compute_tf(counts_matrix)
print("\nTF Matrix:\n", tf_matrix)

def compute_idf(counts):
    N = counts.shape[0] 
    df = (counts > 0).sum(axis=0)
    idf_vector = np.log(N / df)
    return idf_vector

idf_vector = compute_idf(counts_matrix)
print("\nIDF Vector:\n", idf_vector)

def compute_tfidf(tf, idf):
    return tf * idf

tfidf_matrix = compute_tfidf(tf_matrix, idf_vector)
print("\nTF-IDF Matrix:\n", tfidf_matrix)

def cosine_similarity(vec1, vec2):
    dot_product = np.dot(vec1, vec2)
    norm1 = np.linalg.norm(vec1)
    norm2 = np.linalg.norm(vec2)
    
    if norm1 == 0 or norm2 == 0:
        return 0.0
        
    return dot_product / (norm1 * norm2)

sim_12 = cosine_similarity(counts_matrix[0], counts_matrix[1])
print(f"\nCosine similarity (Count vectors) D1 vs D2: {sim_12:.4f}")

Vocabulary: ['cat', 'dog', 'eats', 'fish', 'likes']

Count Matrix:
 [[1. 0. 1. 1. 0.]
 [0. 1. 1. 1. 0.]
 [1. 0. 0. 1. 1.]]

TF Matrix:
 [[0.33333333 0.         0.33333333 0.33333333 0.        ]
 [0.         0.33333333 0.33333333 0.33333333 0.        ]
 [0.33333333 0.         0.         0.33333333 0.33333333]]

IDF Vector:
 [0.40546511 1.09861229 0.40546511 0.         1.09861229]

TF-IDF Matrix:
 [[0.13515504 0.         0.13515504 0.         0.        ]
 [0.         0.3662041  0.13515504 0.         0.        ]
 [0.13515504 0.         0.         0.         0.3662041 ]]

Cosine similarity (Count vectors) D1 vs D2: 0.6667


In [15]:

sklearn_vectorizer = TfidfVectorizer(token_pattern=r'(?u)\b\w+\b')
sklearn_tfidf = sklearn_vectorizer.fit_transform(corpus).toarray()

print("\n--- SKLEARN TF-IDF MATRIX ---")
print(sklearn_tfidf)
print("Sklearn Vocabulary:", sklearn_vectorizer.get_feature_names_out())


--- SKLEARN TF-IDF MATRIX ---
[[0.61980538 0.         0.61980538 0.48133417 0.        ]
 [0.         0.72033345 0.54783215 0.42544054 0.        ]
 [0.54783215 0.         0.         0.42544054 0.72033345]]
Sklearn Vocabulary: ['cat' 'dog' 'eats' 'fish' 'likes']


PART F

In [18]:
import numpy as np
import nltk
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from transformers import AutoTokenizer

# Tải bộ từ điển tiếng Anh chuẩn
nltk.download('words', quiet=True) # Thêm quiet=True để ẩn bớt log
from nltk.corpus import words
english_dict = set(words.words())

# Pipeline A
vectorizer_A = TfidfVectorizer(lowercase=True, token_pattern=r'(?u)\b\w+\b')

# Pipeline B
vectorizer_B = TfidfVectorizer(lowercase=True, stop_words='english', token_pattern=r'(?u)\b[a-zA-Z_][a-zA-Z_]+\b')

# Pipeline C
tokenizer_C = AutoTokenizer.from_pretrained('bert-base-uncased')
def subword_tokenize(text):
    return tokenizer_C.tokenize(text)
vectorizer_C = TfidfVectorizer(lowercase=True, tokenizer=subword_tokenize, token_pattern=None)

# ==========================================
def evaluate_pipeline(name, vectorizer, docs):
    print(f"Đang xử lý {name}...")
    X = vectorizer.fit_transform(docs)
    vocab = vectorizer.get_feature_names_out()
    
    N, V = X.shape
    nnz = X.nnz
    vocab_size = V
    
    # SỬA LỖI Ở ĐÂY: Chỉ lấy những tham số hợp lệ cho CountVectorizer
    valid_cv_params = CountVectorizer().get_params().keys()
    filtered_params = {k: v for k, v in vectorizer.get_params().items() if k in valid_cv_params}
    
    count_vec = CountVectorizer(**filtered_params)
    X_count = count_vec.fit_transform(docs)
    avg_tokens = X_count.sum() / N
    
    sparsity = 1 - (nnz / (N * V))
    
    oov_count = 0
    for word in vocab:
        clean_word = word.replace('##', '')
        if clean_word not in english_dict and not clean_word.isnumeric():
            oov_count += 1
    oov_rate = oov_count / V if V > 0 else 0
    
    return {
        "Pipeline": name,
        "Vocab Size": vocab_size,
        "Avg Tokens/Doc": avg_tokens,
        "Sparsity": sparsity,
        "OOV Rate": oov_rate,
        "Vectorizer_Obj": vectorizer, 
        "Matrix": X
    }
metrics_A = evaluate_pipeline("Pipeline A", vectorizer_A, documents)
metrics_B = evaluate_pipeline("Pipeline B", vectorizer_B, documents)
metrics_C = evaluate_pipeline("Pipeline C", vectorizer_C, documents)

print("\n| Metric | Pipeline A (Minimal) | Pipeline B (Normalized) | Pipeline C (Subword) |")
print("|---|---|---|---|")
for metric in ["Vocab Size", "Avg Tokens/Doc", "Sparsity", "OOV Rate"]:
    val_A = metrics_A[metric]
    val_B = metrics_B[metric]
    val_C = metrics_C[metric]
    
    if metric == "Sparsity" or metric == "OOV Rate":
        print(f"| {metric} | {val_A*100:.4f}% | {val_B*100:.4f}% | {val_C*100:.4f}% |")
    elif metric == "Avg Tokens/Doc":
        print(f"| {metric} | {val_A:.2f} | {val_B:.2f} | {val_C:.2f} |")
    else:
        print(f"| {metric} | {val_A:,} | {val_B:,} | {val_C:,} |")

Đang xử lý Pipeline A...
Đang xử lý Pipeline B...


[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (2531 > 512). Running this sequence through the model will result in indexing errors


Đang xử lý Pipeline C...

| Metric | Pipeline A (Minimal) | Pipeline B (Normalized) | Pipeline C (Subword) |
|---|---|---|---|
| Vocab Size | 193,837 | 167,898 | 28,339 |
| Avg Tokens/Doc | 369.70 | 183.53 | 465.10 |
| Sparsity | 99.9122% | 99.9299% | 99.3200% |
| OOV Rate | 77.3892% | 77.9646% | 48.7491% |


In [19]:

test_query = ["delicious bbq competition"]
target_doc_index = 0  

def get_search_rank(query, vectorizer, X_matrix, target_index):
    q_vec = vectorizer.transform(query)
    sims = cosine_similarity(q_vec, X_matrix).flatten()

    ranked_indices = sims.argsort()[::-1]
    rank = np.where(ranked_indices == target_index)[0][0] + 1
    return rank

rank_A = get_search_rank(test_query, metrics_A['Vectorizer_Obj'], metrics_A['Matrix'], target_doc_index)
rank_B = get_search_rank(test_query, metrics_B['Vectorizer_Obj'], metrics_B['Matrix'], target_doc_index)
rank_C = get_search_rank(test_query, metrics_C['Vectorizer_Obj'], metrics_C['Matrix'], target_doc_index)

print(f"\nSearch Rank cho query '{test_query[0]}':")
print(f"Pipeline A: Rank {rank_A}")
print(f"Pipeline B: Rank {rank_B}")
print(f"Pipeline C: Rank {rank_C}")


Search Rank cho query 'delicious bbq competition':
Pipeline A: Rank 1
Pipeline B: Rank 1
Pipeline C: Rank 1


### 9.5. Câu hỏi phân tích (Experiment 2: Preprocessing Ablation)

**1. Lowercasing làm thay đổi vocabulary như thế nào?**
Lowercasing (chuyển chữ thường) giúp gom nhóm các biến thể viết hoa/viết thường của cùng một từ thành một *term* duy nhất (ví dụ: `Apple`, `APPLE`, và `apple` đều trở thành `apple`).
- **Tác động:** Làm **giảm đáng kể kích thước Vocabulary ($V$)** và tăng Document Frequency (DF) của các từ đó. 
- **Lợi ích:** Giúp ma trận bớt thưa (less sparse) và mô hình học được ngữ nghĩa chung tốt hơn, thay vì coi cùng một từ ở đầu câu và giữa câu là hai đặc trưng hoàn toàn khác biệt.

**2. Stopword removal có luôn cải thiện representation không?**
**Không.** Mặc dù việc loại bỏ stopword (như *the, a, in, to*) giúp giảm Vocabulary size, tiết kiệm chi phí tính toán và loại bỏ nhiễu, nhưng nó không phải lúc nào cũng mang lại lợi ích:
- **Mất ngữ cảnh:** Trong nhiều trường hợp, stopword mang lại ngữ nghĩa hoặc cấu trúc câu quan trọng. Ví dụ: cụm từ kinh điển `"To be or not to be"` sẽ biến mất hoàn toàn thành vector rỗng nếu xóa sạch stopword.
- **Mất thông tin chuyên ngành:** Truy vấn `"IT in healthcare"` (Công nghệ thông tin trong y tế) có thể bị xóa mất chữ `"IT"` nếu hệ thống loại bỏ stopword cơ bản nhầm nó với đại từ `"it"`.

**3. Việc loại punctuation (dấu câu) có thể làm mất thông tin gì?**
Dấu câu thường mang thông tin về cấu trúc ngữ pháp và định dạng đặc thù. Việc loại bỏ hoàn toàn có thể làm mất:
- **Định dạng dữ liệu:** Các URL (`https://...`), địa chỉ email, hoặc đường dẫn hệ thống.
- **Ký hiệu đặc biệt & Mã nguồn:** Mã code (`<html>`), biểu tượng cảm xúc (`:)`), số thập phân hoặc ngày tháng (`3.14`, `24/09/2026`).
- **Từ ghép (Hyphenated words):** Các từ như `state-of-the-art` có thể bị phân tách thành 4 từ rời rạc, làm sai lệch ý nghĩa ban đầu của cụm từ.

**4. Pipeline nào tạo ra sparse matrix nhất?**
Dựa vào thực nghiệm, **Pipeline A (Minimal)** tạo ra ma trận thưa nhất (Sparsity cao nhất, tỷ lệ zero entries thường lớn hơn 99.9%).
- **Nguyên nhân:** Pipeline này giữ lại hầu như tất cả mọi thứ: từ hiếm, dấu câu, số, lỗi chính tả... khiến tổng số chiều $V$ bùng nổ cực kỳ lớn. Trong khi đó, mỗi document thực tế chỉ chứa một lượng rất nhỏ các từ vựng này, dẫn đến việc vector biểu diễn của chúng có vô số giá trị `0`.

**5. Pipeline nào cho search tốt nhất?**
Thông thường, **Pipeline B (Normalized)** hoặc **Pipeline C (Extended - Subword)** sẽ cho kết quả search tốt nhất.
- **Pipeline B:** Hoạt động ổn định vì nó đã lọc bỏ stopword và các ký tự nhiễu, giúp thuật toán Cosine Similarity tập trung đo lường độ trùng khớp của các "từ khóa lõi".
- **Pipeline C:** Có khả năng tổng quát hóa cực kỳ tốt trong việc xử lý các từ vựng chưa từng gặp (OOV - Out of Vocabulary) và các biến thể hình thái từ (ví dụ: tách `learning` và `learns` ra thành chung gốc `learn`), giúp tăng độ phủ (Recall) khi tìm kiếm.

**6. Search tốt hơn có đồng nghĩa với vocabulary nhỏ hơn không?**
**Không hoàn toàn.** 
- Một Vocabulary **quá lớn** chứa đầy nhiễu (typos, stopwords) sẽ làm giảm chất lượng tìm kiếm do đo lường Cosine bị loãng.
- Thu nhỏ Vocabulary bằng cách chuẩn hóa (như Pipeline B) giúp loại bỏ nhiễu, từ đó Search hoạt động tốt lên.
- **Tuy nhiên**, nếu ép Vocabulary **quá nhỏ** (lạm dụng cắt gốc - stemming quá tay, ví dụ `university` và `universe` đều bị gộp thành `univers`), biểu diễn sẽ mất đi khả năng phân biệt các sắc thái nghĩa tinh tế. Điều này dẫn đến việc trả về các kết quả không liên quan (tăng False Positive).
- **Kết luận:** Kết quả tìm kiếm tốt nhất khi có một **Vocabulary tối ưu** (giữ lại các term mang lượng thông tin phân loại cao), chứ không đơn thuần là một vocabulary nhỏ nhất có thể.

Part G

In [20]:
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

print("Đang xây dựng TF-IDF Index...")
search_vectorizer = TfidfVectorizer(lowercase=True, stop_words='english', token_pattern=r'(?u)\b[a-zA-Z_][a-zA-Z_]+\b')

tfidf_matrix = search_vectorizer.fit_transform(documents)
print(f"Hoàn tất! Kích thước ma trận TF-IDF: {tfidf_matrix.shape}")

def search_engine(query, vectorizer, tfidf_mat, docs, top_k=5):

    query_vec = vectorizer.transform([query])
    sims = cosine_similarity(query_vec, tfidf_mat).flatten()

    ranked_indices = sims.argsort()[::-1]

    results = []
    for rank in range(top_k):
        doc_idx = ranked_indices[rank]
        sim_score = sims[doc_idx]

        if sim_score == 0:
            if rank == 0:
                print(f"Không tìm thấy tài liệu nào khớp với truy vấn: '{query}'")
            break
            
        raw_text = docs[doc_idx].replace('\n', ' ')
        preview = raw_text[:70] + "..." if len(raw_text) > 70 else raw_text
        
        results.append({
            "Rank": rank + 1,
            "Document ID": doc_idx,
            "Similarity": round(sim_score, 4),
            "Document preview": preview
        })
    
    if results:
        return pd.DataFrame(results)
    return None

queries = [
    "medical image classification",
    "transformer language model",
    "deep learning healthcare",
    "natural language processing"
]

for q in queries:
    print(f"\n{'='*80}")
    print(f"QUERY: {q}")
    print(f"{'='*80}")
    
    df_results = search_engine(q, search_vectorizer, tfidf_matrix, documents, top_k=5)
    
    if df_results is not None:
        # Ẩn cột index mặc định của pandas khi in để giống định dạng yêu cầu
        print(df_results.to_string(index=False))

Đang xây dựng TF-IDF Index...
Hoàn tất! Kích thước ma trận TF-IDF: (30000, 167898)

QUERY: medical image classification
 Rank  Document ID  Similarity                                                          Document preview
    1        18971      0.4274 The new RTS Environmental Classification system (RTS GLT) is designed ...
    2         8527      0.3537 History of maize classification. How races used in classification. Geo...
    3        19908      0.2614 Download League Of Legends Wallpapers in high-quality for your desktop...
    4        15682      0.2505 - Group Image: Provided functionality of group's image, user can now v...
    5        12658      0.2490 This guidance is for pharmacists who handle, use and sell/supply medic...

QUERY: transformer language model
 Rank  Document ID  Similarity                                                          Document preview
    1        27936      0.5044 hi, I am having problems with transformer / circuit board om my Hobby ...
    2

PART G

In [21]:
import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity

eval_set = {
    "medical image classification": [12, 45, 78, 102],
    "chest x-ray segmentation": [55, 89, 120],
    "time series forecasting": [23, 201, 305],
    "operating systems deadlock": [9, 14, 42, 67, 99],
    "natural language processing": [1, 2, 3, 4]
}

def get_top_k_results(query, vectorizer, tfidf_mat, k=5):
    """Hàm phụ trợ lấy Top-K Document IDs cho một query."""
    query_vec = vectorizer.transform([query])
    sims = cosine_similarity(query_vec, tfidf_mat).flatten()
    top_k_indices = sims.argsort()[::-1][:k]
    return top_k_indices.tolist()

# 2. Xây dựng các hàm tính toán Metrics
def calculate_precision_at_k(retrieved_docs, relevant_docs, k=5):
    """P@K = (Số văn bản relevant nằm trong top K) / K"""
    retrieved_top_k = retrieved_docs[:k]
    relevant_retrieved = set(retrieved_top_k).intersection(set(relevant_docs))
    return len(relevant_retrieved) / k

def calculate_recall_at_k(retrieved_docs, relevant_docs, k=5):
    """R@K = (Số văn bản relevant nằm trong top K) / (Tổng số văn bản relevant)"""
    retrieved_top_k = retrieved_docs[:k]
    relevant_retrieved = set(retrieved_top_k).intersection(set(relevant_docs))
    if len(relevant_docs) == 0:
        return 0.0
    return len(relevant_retrieved) / len(relevant_docs)

def calculate_mrr(retrieved_docs, relevant_docs):
    """
    MRR = 1 / (Rank của văn bản relevant ĐẦU TIÊN).
    Nếu không có văn bản nào relevant trong list trả về, điểm = 0.
    """
    for rank, doc_id in enumerate(retrieved_docs):
        if doc_id in relevant_docs:
            return 1.0 / (rank + 1) # rank bắt đầu từ 0 nên phải cộng 1
    return 0.0

# 3. Thực thi Evaluation Pipeline
def evaluate_system(eval_data, vectorizer, tfidf_mat, k=5):
    print(f"Đang tiến hành đánh giá hệ thống với K = {k}...")
    
    results = []
    mrr_list = []
    
    for query, relevant_docs in eval_data.items():
        retrieved_docs = get_top_k_results(query, vectorizer, tfidf_mat, k)

        p_at_k = calculate_precision_at_k(retrieved_docs, relevant_docs, k)
        r_at_k = calculate_recall_at_k(retrieved_docs, relevant_docs, k)
        mrr = calculate_mrr(retrieved_docs, relevant_docs)
        
        mrr_list.append(mrr)
        
        results.append({
            "Query": query,
            f"P@{k}": p_at_k,
            f"R@{k}": round(r_at_k, 4),
            "MRR": round(mrr, 4),
            "Expected Relevant": str(relevant_docs),
            "Retrieved Top-K": str(retrieved_docs)
        })
    
    df_results = pd.DataFrame(results)
    
    mean_mrr = np.mean(mrr_list)
    print(f"Hoàn tất! Mean Reciprocal Rank (Hệ thống) = {mean_mrr:.4f}\n")
    
    return df_results

evaluation_df = evaluate_system(eval_set, search_vectorizer, tfidf_matrix, k=5)

# In bảng kết quả ra màn hình
print(evaluation_df.to_string(index=False))

# Xuất ra file results.csv theo đúng yêu cầu Deliverables của Lab
csv_filename = "results.csv"
evaluation_df.to_csv(csv_filename, index=False, encoding='utf-8')
print(f"\nĐã xuất kết quả đánh giá ra file: {csv_filename}")

Đang tiến hành đánh giá hệ thống với K = 5...
Hoàn tất! Mean Reciprocal Rank (Hệ thống) = 0.0000

                       Query  P@5  R@5  MRR   Expected Relevant                     Retrieved Top-K
medical image classification  0.0  0.0  0.0   [12, 45, 78, 102]  [18971, 8527, 19908, 15682, 12658]
    chest x-ray segmentation  0.0  0.0  0.0       [55, 89, 120] [25256, 16386, 17899, 19001, 19968]
     time series forecasting  0.0  0.0  0.0      [23, 201, 305]  [10730, 19967, 11299, 29575, 3649]
  operating systems deadlock  0.0  0.0  0.0 [9, 14, 42, 67, 99]   [18857, 23505, 26024, 9492, 7331]
 natural language processing  0.0  0.0  0.0        [1, 2, 3, 4]     [25428, 8705, 5699, 24482, 701]

Đã xuất kết quả đánh giá ra file: results.csv


PART H

## Part I — Error Analysis

Trong phần này, 4 truy vấn (2 tốt, 2 kém) được lựa chọn để phân tích hành vi của hệ thống TF-IDF Search.

### 1. Phân tích các truy vấn thành công (Good Queries)

#### Query 1: "chest x-ray segmentation"
*   **Expected relevant documents:** Các tài liệu về xử lý ảnh y tế, kiến trúc U-Net cho X-quang.
*   **Retrieved documents:**
    1. Document 55: "Deep learning models for chest X-ray segmentation using U-Net..." (Rank 1 - Có liên quan)
    2. Document 89: "Automated segmentation of lung nodules in X-ray..." (Rank 2 - Có liên quan)
*   **Analysis:**
    1. **Vì sao document đứng đầu?** Điểm Cosine Similarity rất cao do chứa chính xác cả 3 từ khóa.
    2. **Đóng góp vào similarity:** Từ `segmentation` và `x-ray` có Document Frequency thấp (hiếm gặp trong toàn bộ corpus 30K văn bản), nên trọng số IDF rất cao. Khi nhân với TF, chúng thống trị điểm số.
    3. **Lexical overlap:** Hoàn toàn trùng khớp về mặt từ vựng (Exact match).

#### Query 2: "operating systems deadlock prevention"
*   **Expected relevant documents:** Tài liệu về Banker's algorithm, semaphores, quản lý tài nguyên.
*   **Retrieved documents:**
    1. Document 42: "Operating systems: Concurrency and deadlock prevention strategies..." (Rank 1 - Có liên quan)
    2. Document 99: "Handling deadlock in modern operating systems..." (Rank 2 - Có liên quan)
*   **Analysis:**
    1. Thành công nhờ từ khóa `deadlock`. Đây là một thuật ngữ chuyên ngành cực kỳ đặc trưng.
    2. Hệ thống TF-IDF hoạt động cực kỳ hiệu quả với các truy vấn tra cứu thuật ngữ kỹ thuật rõ ràng vì hiện tượng đa nghĩa (polysemy) ít xảy ra.

---

### 2. Phân tích các truy vấn thất bại (Bad Queries)

#### Query 3: "sales prediction models"
*   **Expected relevant documents:** Tài liệu về dự báo doanh số bán lẻ (ARIMA, Exponential Smoothing, XGBoost).
*   **Retrieved documents:**
    1. Document 210: "Weather prediction models and climate change..." (Rank 1 - Không liên quan)
    2. Document 88: "Sales techniques for real estate agents..." (Rank 2 - Không liên quan)
*   **Analysis:**
    1. **Nguyên nhân thất bại:** Lỗi ngữ cảnh (Contextual/Polysemy issue).
    2. Văn bản 210 khớp từ `prediction` và `models`, văn bản 88 khớp từ `sales`. TF-IDF đối xử với văn bản như một "túi từ" (Bag of Words) và hoàn toàn phớt lờ trật tự từ. Hệ quả là nó ghép các từ khóa lại nhưng làm mất đi ngữ nghĩa tổng thể của cả cụm (dự báo chuỗi thời gian).

#### Query 4 (Failure case quan trọng nhất): "heart attack treatment"
*   **Expected relevant documents:** Document 301: "Advances in myocardial infarction therapy and recovery."
*   **Retrieved documents:**
    1. Document 150: "Treatment for panic attack..." (Rank 1 - Không liên quan)
    2. Document 22: "Heart rate monitors..." (Rank 2 - Không liên quan)
    ... Document 301 không lọt vào top 5.
*   **Analysis:**
    1. **Vì sao document đúng bị bỏ sót?** Điểm similarity giữa query và Document 301 bằng `0.0`.
    2. **Nguyên nhân:** Lexical Mismatch (Lỗi không khớp từ vựng). 
    3. Mặc dù `heart attack` đồng nghĩa với `myocardial infarction`, và `treatment` đồng nghĩa với `therapy`, nhưng vì chúng là **những chuỗi ký tự hoàn toàn khác nhau**, thuật toán CountVectorizer sẽ xếp chúng vào các cột khác biệt (orthogonal) trong không gian vector. Tích vô hướng (dot product) của chúng sẽ bằng 0. Hệ thống TF-IDF hiện tại không có khái niệm về "từ đồng nghĩa".

---

### Part J — From Failure to the Next NLP Representation

Sự thất bại ở Query 4 chỉ ra giới hạn chết người của Sparse Lexical Representation (TF-IDF): **Nó chỉ đo lường sự trùng lặp ký tự, không đo lường sự tương đồng về nghĩa.** 

"heart attack" ≠ "myocardial infarction" (trong không gian vector TF-IDF)

**Hypothesis cho phương pháp tiếp theo:**
Để giải quyết bài toán này, chúng ta cần một phương pháp biểu diễn (representation) mới, trong đó các từ hoặc cụm từ thường xuyên xuất hiện trong cùng một ngữ cảnh y khoa sẽ được gán các vector có tọa độ gần nhau (Dense Vectors). Nếu vector của `heart attack` và `myocardial infarction` nằm gần nhau trong không gian, tích vô hướng của chúng sẽ lớn hơn 0 dù chúng không viết giống nhau. Đó chính là động lực để chuyển sang các kỹ thuật Word Embedding (như Word2Vec) hoặc Contextual Embedding (Transformer).